In [2]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# ============================================================
# 0. 데이터 불러오기
# ============================================================

train_full = pd.read_parquet(PROJECT_ROOT / 'data' / '2_train_full_fe.parquet')
test = pd.read_parquet(PROJECT_ROOT / 'data' / '2_test_fe.parquet')

In [4]:
# ============================================================
# 1. 컬럼군 정의
# ============================================================

target_col = 'target_maxrise_6h'  # 모델링 학습부터는 반드시 빼내야 함

# 시계열 컬럼군
time_cols = [
    'wl_t',          # 수위데이터 대표
    'sfc_rain_1h',   # 육상 누적강수데이터 대표
    'imerg_rain_1h', # 위성 누적강수데이터 대표
    'radar_reflectivity_mean', # 레이더반사도 대표
    'radar_reflectivity_p95', # 레이더반사도 대표
]

# 수위 컬럼군
wl_cols = [
    'wl_t',
    'wl_t_minus_1h',
    'wl_t_minus_3h',
    'wl_t_minus_6h',
    'wl_t_minus_12h',
    'wl_t_minus_24h',
]

# 수위차분(wl_diff) 컬럼군
wl_diff_cols = [
    'wl_t',
    'wl_diff_1h',
    'wl_diff_3h',
    'wl_diff_6h',
    'wl_diff_12h',
    'wl_diff_24h'
]


# 강수 컬럼군
rain_cols = [
    # IMERG강우
    'imerg_rain_1h',
    'imerg_rain_6h',
    'imerg_rain_24h',
    # 지상강우
    'sfc_rain_1h',
    'sfc_rain_6h',
    'sfc_rain_24h',
    # 레이더 반사도
    'radar_reflectivity_mean',
    'radar_reflectivity_p95',
    'radar_reflectivity_ge20_fraction'
]

# 강우강도 컬럼군
rain_int_cols = [
    'imerg_rain_1h',
    'imerg_int_1to6h',
    'imerg_int_6to24h',
    'sfc_rain_1h',
    'sfc_int_1to6h',
    'sfc_int_6to24h'
]

# 환경 컬럼군
env_cols = [
    'cat_area',       # km²
    'wsarea',         # km²
    'mean_elevation', # m
    'mean_slope',     # °
    'urban_ratio',    # %
    'forest_ratio',   # %
    'stream_order'
]

num_cols = wl_cols + rain_cols + env_cols
num_cols_fe_wl = num_cols + [
    col for col in wl_diff_cols if col != 'wl_t'
]
num_cols_fe = num_cols_fe_wl + [
    col for col in rain_int_cols if col not in ['imerg_rain_1h', 'sfc_rain_1h']
]

print(f"""<변수개수 검증>
    수위변수: {len(wl_cols)}
    강우변수: {len(rain_cols)}, 
    환경변수: {len(env_cols)}, 
    수위차분(wl_t포함): {len(wl_diff_cols)},
    강우강도(rain_1h포함): {len(rain_int_cols)}
    기존 숫자열(수위 + 강우 + 환경): {len(num_cols)}, 
    수위차분추가 숫자열(수위 + 강우 + 환경 + 수위차분(wl_t제외)): {len(num_cols_fe_wl)},
    강우강도추가 숫자열(수위차분추가 숫자열 + 강우강도(rain_1h 2개제외)): {len(num_cols_fe)}
""")

<변수개수 검증>
    수위변수: 6
    강우변수: 9, 
    환경변수: 7, 
    수위차분(wl_t포함): 6,
    강우강도(rain_1h포함): 6
    기존 숫자열(수위 + 강우 + 환경): 22, 
    수위차분추가 숫자열(수위 + 강우 + 환경 + 수위차분(wl_t제외)): 27,
    강우강도추가 숫자열(수위차분추가 숫자열 + 강우강도(rain_1h 2개제외)): 31



In [5]:
# ============================================================
# 3. 이상치(계측오류) 제거: train_full 종속변수 이상치 1개 제거
# ============================================================
# 변수별 분포특성 확인단계에서 확인됨

# 제거 조건 설정
cond_maxrise_outlier = train_full['target_maxrise_6h'] < -8  # 이거 자체가 불리언 조건문임
print(f" cond_maxrise_outlier 조건에 해당하는 데이터 개수: {cond_maxrise_outlier.sum()}")  # 불리언조건문은 sum으로 개수를 셀 수 있음
print(f"이상치 행 제거 전 데이터 개수: {len(train_full)}") 

# 제거 대상 지정
print()
maxrise_outlier_row_id = (
    train_full
    .loc[cond_maxrise_outlier]
    .index
)
print(maxrise_outlier_row_id)

# 이상치 제거
train_full = train_full.drop(index = maxrise_outlier_row_id)
print(f"이상치 행 제거 후 train셋 행 개수: {len(train_full)}")
print()

 cond_maxrise_outlier 조건에 해당하는 데이터 개수: 1
이상치 행 제거 전 데이터 개수: 749657

Index(['TR_000583385'], dtype='str', name='row_id')
이상치 행 제거 후 train셋 행 개수: 749656



In [6]:
# ============================================================
# 4. 이상치(계측오류) 제거: 수위차분변수-종속변수 산점도 이상치
# ============================================================
# 변수간 상관계수 분석에서 확인됨

# 제거 조건 설정
cond_wl_out_ST1 = train_full['station_id'] == 'ST0445'
cond_wl_out_ST2 = train_full['station_id'] == 'ST0066'
cond_wl_out_ST0445 = (train_full[wl_cols] > 15).any(axis=1)
cond_wl_out_ST0066 = (train_full[wl_cols] > 23).any(axis=1)
cond_wl_f = (
    (cond_wl_out_ST1 & cond_wl_out_ST0445)|
    (cond_wl_out_ST2 & cond_wl_out_ST0066)
    )

print(
    f"ccond_wl_f 조건에 해당하는 데이터 개수: "
    f"{cond_wl_f.sum()}"
)  # 불리언조건문은 sum으로 개수를 셀 수 있음
print(f"이상치 행 제거 전 데이터 개수: {len(train_full)}") 

# 제거 대상 지정
print()
wl_outlier_row_id = (
    train_full
    .loc[cond_wl_f]
    .index
)

# 이상치 제거
train_full = train_full.drop(index = wl_outlier_row_id)
print(f"이상치 행 제거 후 train셋 행 개수: {len(train_full)}")
print()


ccond_wl_f 조건에 해당하는 데이터 개수: 32
이상치 행 제거 전 데이터 개수: 749656

이상치 행 제거 후 train셋 행 개수: 749624



In [15]:
# ============================================================
# 5. 이상치(계측오류) 제거: 수위차분변수 이상치 제거
# ============================================================
# train, test셋 분포 비교에서 확인

# 제거 조건 설정
cond_wl_out_ST3 = train_full['station_id'] == 'ST0089'
cond_wl_out_ST4 = train_full['station_id'] == 'ST0158'
cond_wl_out_ST5 = train_full['station_id'] == 'ST0448'
cond_wl_out_ST6 = train_full['station_id'] == 'ST0007'
cond_wl_out_ST0089 = (train_full[wl_cols] > 6).any(axis=1)
cond_wl_out_ST0158 = (train_full[wl_cols] > 7).any(axis=1)
cond_wl_out_ST0448 = (
    (train_full[wl_cols] > 9).any(axis=1) &
    (train_full.index != 'TR_000199803')
)
cond_wl_out_ST0007 = (train_full[wl_cols] > 8).any(axis=1)
cond_wl_f2 = (
    (cond_wl_out_ST3 & cond_wl_out_ST0089)|
    (cond_wl_out_ST4 & cond_wl_out_ST0158)|
    (cond_wl_out_ST5 & cond_wl_out_ST0448)|
    (cond_wl_out_ST6 & cond_wl_out_ST0007)
    )

print(
    f"ccond_wl_f 조건에 해당하는 데이터 개수: "
    f"{cond_wl_f2.sum()}"
)  # 불리언조건문은 sum으로 개수를 셀 수 있음
print(f"이상치 행 제거 전 데이터 개수: {len(train_full)}") 

# 제거 대상 지정
print()
wl_outlier2_row_id = (
    train_full
    .loc[cond_wl_f2]
    .index
)
print(wl_outlier2_row_id)

# 이상치 제거
train_full = train_full.drop(index = wl_outlier2_row_id)
print(f"이상치 행 제거 후 train셋 행 개수: {len(train_full)}")
print()


ccond_wl_f 조건에 해당하는 데이터 개수: 15
이상치 행 제거 전 데이터 개수: 749624

Index(['TR_000547855', 'TR_000457689', 'TR_000318564', 'TR_000193482',
       'TR_000057205', 'TR_000154692', 'TR_000715299', 'TR_000468355',
       'TR_000450715', 'TR_000267036', 'TR_000476408', 'TR_000185855',
       'TR_000385460', 'TR_000665570', 'TR_000480061'],
      dtype='str', name='row_id')
이상치 행 제거 후 train셋 행 개수: 749609



In [ ]:
# ======================================================================
# 6. 이상치 제거 후, parquet 저장
# ======================================================================

train_full.to_parquet(PROJECT_ROOT / 'data' / '3_train_full_clean.parquet')